In [ ]:
import random
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers


# -------------------------------------------------------------
# 0. Establish SEED (REPRODUCIBILITY)
# -------------------------------------------------------------
def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  tf.random.set_seed(seed)


set_seed(42)

# -------------------------------------------------------------
# 1. LOAD AND PRE-PROCESSING FILE .NPZ
# -------------------------------------------------------------

npz_path = r'./npz/dataset2025_ready2.npz'
data = np.load(npz_path)

print(f'Các khóa có trong file .npz: {list(data.keys())}')

X_tr_raw = data['X_train']
y_tr_raw = data['y_train']
X_te_raw = data['X_test']
y_te_raw = data['y_test']

# Read poly_ids from Notebook 1
if 'poly_ids_train' in data:
  poly_ids = data['poly_ids_train']
elif 'poly_ids' in data:
  poly_ids = data['poly_ids']
else:
  raise KeyError(
      "No found 'poly_ids_train' or 'poly_ids' trong file .npz!"
  )

# Convert labels into 0-indexed [0, num_classes - 1]
min_label = min(y_tr_raw.min(), y_te_raw.min())
y_tr = y_tr_raw - min_label
y_te = y_te_raw - min_label

num_classes = len(np.unique(y_tr))
class_labels = [
    'Mangrove',
    'Mixed mangrove',
    'Aquaculture',
    'Built_up',
    'Open water',
]

# Convert Channels-Last of Keras: (N, H, W, C) từ X_tr_raw / X_te_raw
if X_tr_raw.ndim == 4 and X_tr_raw.shape[1] < X_tr_raw.shape[2]:
  X_train_full = np.transpose(X_tr_raw, (0, 2, 3, 1))
  X_test_full = np.transpose(X_te_raw, (0, 2, 3, 1))
else:
  X_train_full = X_tr_raw
  X_test_full = X_te_raw

# Remove NDVI (at tha last band -1)
ndvi_index = -1
X_train_no_ndvi = np.delete(X_train_full, ndvi_index, axis=-1)
X_test_no_ndvi = np.delete(X_test_full, ndvi_index, axis=-1)

print(
    f'\nFull (with NDVI) : Train {X_train_full.shape} | Test'
    f' {X_test_full.shape}'
)
print(
    f'without NDVI        : Train {X_train_no_ndvi.shape} | Test'
    f' {X_test_no_ndvi.shape}'
)
print(f'Số lượng Polygon ID (Groups)  : {len(poly_ids)}')


# -------------------------------------------------------------
# 2. MODEL DEFINITION OF CNN KERAS
# -------------------------------------------------------------
def build_cnn_pca(input_shape, num_classes, l2_reg=1e-4, dropout_rate=0.3):
  model = models.Sequential([
      layers.Input(shape=input_shape),
      # Block 1
      layers.Conv2D(
          32,
          (3, 3),
          padding='same',
          kernel_regularizer=regularizers.l2(l2_reg),
      ),
      layers.BatchNormalization(),
      layers.Activation('relu'),
      layers.MaxPooling2D((2, 2)),
      # Block 2
      layers.Conv2D(
          64,
          (3, 3),
          padding='same',
          kernel_regularizer=regularizers.l2(l2_reg),
      ),
      layers.BatchNormalization(),
      layers.Activation('relu'),
      layers.MaxPooling2D((2, 2)),
      # Block 3
      layers.Conv2D(
          128,
          (3, 3),
          padding='same',
          kernel_regularizer=regularizers.l2(l2_reg),
      ),
      layers.BatchNormalization(),
      layers.Activation('relu'),
      # Feature Aggregation & Classifier
      layers.GlobalAveragePooling2D(),
      layers.Dense(64, kernel_regularizer=regularizers.l2(l2_reg)),
      layers.BatchNormalization(),
      layers.Activation('relu'),
      layers.Dropout(dropout_rate),
      layers.Dense(num_classes, activation='softmax'),
  ])

  model.compile(
      optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
      loss='sparse_categorical_crossentropy',
      metrics=['accuracy'],
  )
  return model


# -------------------------------------------------------------
# 3. TRAIN AND EVALUATION
# -------------------------------------------------------------
def train_and_eval_keras(
    X_train, y_train, X_test, y_test, num_classes, groups=None, name='CNN', save_name="cnn_model"
):
  print('==================================================')
  print(f'\n---> TRAINING Keras CNN: {name}...')
  print('==================================================')

  input_shape = X_train.shape[1:]

  # 1. CROSS-VALIDATION (5-FOLD SPATIAL GROUP)
  sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
  cv_accs, cv_macro_f1s, cv_weighted_f1s = [], [], []

  y_train_labels = np.argmax(y_train, axis=1) if y_train.ndim > 1 else y_train

  # Split data based on poly_ids
  for fold, (train_idx, val_idx) in enumerate(
      sgkf.split(X_train, y_train_labels, groups=groups)
  ):
    set_seed(42)

    X_tr_f, y_tr_f = X_train[train_idx], y_train[train_idx]
    X_va_f, y_va_f = X_train[val_idx], y_train[val_idx]

    fold_model = build_cnn_pca(input_shape, num_classes)

    callbacks_cv = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=7, restore_best_weights=True
        )
    ]

    fold_model.fit(
        X_tr_f,
        y_tr_f,
        validation_data=(X_va_f, y_va_f),
        epochs=30,
        batch_size=64,
        callbacks=callbacks_cv,
        verbose=0,
    )

    val_preds_prob = fold_model.predict(X_va_f, verbose=0)
    val_preds = np.argmax(val_preds_prob, axis=1)
    y_va_true = np.argmax(y_va_f, axis=1) if y_va_f.ndim > 1 else y_va_f

    cv_accs.append(accuracy_score(y_va_true, val_preds))
    cv_macro_f1s.append(f1_score(y_va_true, val_preds, average='macro'))
    cv_weighted_f1s.append(f1_score(y_va_true, val_preds, average='weighted'))

  cv_oa = np.mean(cv_accs) * 100
  cv_oa_std = np.std(cv_accs) * 100
  cv_f1_macro = np.mean(cv_macro_f1s) * 100
  cv_f1_weighted = np.mean(cv_weighted_f1s) * 100

  print(f'[CV 5-Fold Train] Mean OA        : {cv_oa:.2f}% (± {cv_oa_std:.2f}%)')
  print(f'[CV 5-Fold Train] Mean Macro F1 : {cv_f1_macro:.2f}%')
  print(f'[CV 5-Fold Train] Mean Weight F1: {cv_f1_weighted:.2f}%\n')

  # 2. MODEL TRAINING/VALIDATION AND TEST EVALUATION
  
  # StratifiedGroupKFold creates a fold validation (~20%).
  final_splitter = StratifiedGroupKFold(
      n_splits=5, shuffle=True, random_state=42
  )
  fit_idx, val_idx = next(
      final_splitter.split(X_train, y_train_labels, groups=groups)
  )
  if np.intersect1d(groups[fit_idx], groups[val_idx]).size:
    raise RuntimeError('Polygon ID overlapped train-fit and validation.')
  print(
      f'[Final fit] Train: {len(fit_idx)} | Validation: {len(val_idx)}'
      f' | Test (chỉ đánh giá cuối): {len(X_test)}'
  )

  set_seed(42)
  final_model = build_cnn_pca(input_shape, num_classes)
  callbacks = [
      tf.keras.callbacks.EarlyStopping(
          monitor='val_loss', patience=7, restore_best_weights=True
      )
  ]

  final_model.fit(
      X_train[fit_idx],
      y_train[fit_idx],
      validation_data=(X_train[val_idx], y_train[val_idx]),
      epochs=30,
      batch_size=64,
      callbacks=callbacks,
      verbose=0,
  )

  # Test
  preds_prob = final_model.predict(X_test, verbose=0)
  preds = np.argmax(preds_prob, axis=1)

  y_test_true = np.argmax(y_test, axis=1) if y_test.ndim > 1 else y_test

  test_oa = accuracy_score(y_test_true, preds) * 100
  test_kappa = cohen_kappa_score(y_test_true, preds)
  test_f1_macro = f1_score(y_test_true, preds, average='macro') * 100
  test_f1_weighted = f1_score(y_test_true, preds, average='weighted') * 100
  f1_per_class = f1_score(y_test_true, preds, average=None) * 100

  print('[Test Set Evaluation]')
  print(f'  - Overall Accuracy (OA) : {test_oa:.2f}%')
  print(f"  - Cohen's Kappa (κ)     : {test_kappa:.4f}")
  print(f'  - Macro F1-Score        : {test_f1_macro:.2f}%')
  print(f'  - Weighted F1-Score     : {test_f1_weighted:.2f}%\n')

  results = {
      'Model': name,
      'CV_OA (%)': f'{cv_oa:.2f} ± {cv_oa_std:.2f}',
      'Test_OA (%)': test_oa,
      'Test_Kappa': test_kappa,
      'Test_Macro_F1 (%)': test_f1_macro,
      'Test_Weighted_F1 (%)': test_f1_weighted,
      'y_pred': preds,
      'f1_per_class': f1_per_class,
  }
 
  final_model.save(f"{save_name}.keras")
  print(f"Saved mdel at: {save_name}.keras")
  return test_oa, preds, results

# -------------------------------------------------------------
# 4. Running model
# -------------------------------------------------------------

# Running 1: with NDVI
acc_full, preds_full, res_full = train_and_eval_keras(
    X_train_full,
    y_tr,
    X_test_full,
    y_te,
    num_classes,
    groups=poly_ids,
    name='CNN (Full Bands - Có NDVI)',
    save_name="cnn_model_4bands"
)

# Running 2: without NDVI
acc_no_ndvi, preds_no_ndvi, res_no_ndvi = train_and_eval_keras(
    X_train_no_ndvi,
    y_tr,
    X_test_no_ndvi,
    y_te,
    num_classes,
    groups=poly_ids,
    name='CNN (without NDVI)',
    save_name="cnn_model_3bands"
)

In [ ]:
import numpy as np
import pandas as pd

# =============================================================
# 4. REPORT AND ACCURACY COMPARISION
# =============================================================
# 4.1. Accuracy comparision
summary_data = {
    'Metric / Indicator': [
        '5-Fold CV Accuracy (%)',
        'Test Overall Accuracy - OA (%)',
        'Test Kappa Coefficient (κ)',
        'Test Macro F1-Score (%)',
        'Test Weighted F1-Score (%)',
    ],
    'CNN Without NDVI': [
        res_no_ndvi['CV_OA (%)'],
        f"{res_no_ndvi['Test_OA (%)']:.2f}",
        f"{res_no_ndvi['Test_Kappa']:.4f}",
        f"{res_no_ndvi['Test_Macro_F1 (%)']:.2f}",
        f"{res_no_ndvi['Test_Weighted_F1 (%)']:.2f}",
    ],
    'CNN With NDVI': [
        res_full['CV_OA (%)'],
        f"{res_full['Test_OA (%)']:.2f}",
        f"{res_full['Test_Kappa']:.4f}",
        f"{res_full['Test_Macro_F1 (%)']:.2f}",
        f"{res_full['Test_Weighted_F1 (%)']:.2f}",
    ],
    'Difference (With - Without)': [
        '-',
        f"{res_full['Test_OA (%)'] - res_no_ndvi['Test_OA (%)']:+.2f}%",
        f"{res_full['Test_Kappa'] - res_no_ndvi['Test_Kappa']:+.4f}",
        (
            f"{res_full['Test_Macro_F1 (%)'] - res_no_ndvi['Test_Macro_F1 (%)']:+.2f}%"
        ),
        (
            f"{res_full['Test_Weighted_F1 (%)'] - res_no_ndvi['Test_Weighted_F1 (%)']:+.2f}%"
        ),
    ],
}

df_summary = pd.DataFrame(summary_data)
print('\n==================================================')
print('   Accuracy comparision (TEST SET)')
print('==================================================')
print(df_summary.to_string(index=False))

# 4.2. Class-per F1-Score
class_labels = [
    'Mangrove',
    'Mixed mangrove',
    'Aquaculture',
    'Built_up',
    'Open water',
]

df_f1_classes = pd.DataFrame({
    'Class Name': class_labels,
    'F1 Without NDVI (%)': np.round(res_no_ndvi['f1_per_class'], 2),
    'F1 With NDVI (%)': np.round(res_full['f1_per_class'], 2),
    'Gain / Loss (%)': np.round(
        res_full['f1_per_class'] - res_no_ndvi['f1_per_class'], 2
    ),
})

print('\n==================================================')
print('   F1-score per class')
print('==================================================')
print(df_f1_classes.to_string(index=False))

# 4.3. Xuất kết quả ra file Excel
output_excel_path = 'CNN_NDVI_Comparison.xlsx'
with pd.ExcelWriter(output_excel_path) as writer:
  df_summary.to_excel(writer, sheet_name='Overall_Metrics', index=False)
  df_f1_classes.to_excel(writer, sheet_name='Per_Class_F1', index=False)

print(f"\n[OK] Successful exported Excel file for comparision: {output_excel_path}")

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np

# =============================================================
# 5. SIMULATION (FIGURE)
# =============================================================
# Font and size of figure
plt.figure(figsize=(9, 5), dpi=300)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'  # or 'Times New Roman'
plt.rcParams['font.size'] = 11

x = np.arange(len(class_labels))
width = 0.35  


f1_no_ndvi = res_no_ndvi['f1_per_class']
f1_full = res_full['f1_per_class']

plt.bar(
    x - width / 2,
    f1_no_ndvi,
    width,
    label='CNN Without NDVI',
    color='#4C72B0',
    edgecolor='black',
    linewidth=0.5,
)
plt.bar(
    x + width / 2,
    f1_full,
    width,
    label='CNN With NDVI',
    color='#55A868',
    edgecolor='black',
    linewidth=0.5,
)

plt.ylabel('F1-Score (%)', fontsize=11, fontweight='bold')
plt.title(
    'Per-Class F1-Score Comparison: With vs. Without NDVI',
    fontsize=12,
    fontweight='bold',
    pad=12,
)
plt.xticks(x, class_labels, rotation=15, ha='right')
plt.ylim(0, 108) 
plt.legend(loc='lower right', frameon=True)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Display values in columne of figure
for i in range(len(class_labels)):
  plt.text(
      i - width / 2,
      f1_no_ndvi[i] + 1.2,
      f'{f1_no_ndvi[i]:.1f}',
      ha='center',
      va='bottom',
      fontsize=8.5,
      fontweight='bold',
  )
  plt.text(
      i + width / 2,
      f1_full[i] + 1.2,
      f'{f1_full[i]:.1f}',
      ha='center',
      va='bottom',
      fontsize=8.5,
      fontweight='bold',
  )

plt.tight_layout()
plt.savefig('NDVI_Impact_Per_Class_F1.png', dpi=300)
plt.show()

print("[OK] Successful exported figure: NDVI_Impact_Per_Class_F1.png")